### Sudoku-Solver

Beim [Sudoku](https://de.wikipedia.org/wiki/Sudoku)
sind Zahlen in ein 9x9 Gitter zu setzen.
Dabei darf jede der Zahlen 1-9 in jeder Reihe, Spalte oder 3x3-Block höchstens einmal vorkommen.
Ist diese Bedingung erfüllt, sprechen wir von einer **Teillösungen**.
Eine korrekt gestelltes Sudoku ist eine Teillösung, welches genau eine Lösung besitzt.
 
<img src='/files/images/250px_Sudoku_mit_Kandidaten.png'>


Ein Sudoku kann man wie das Damenproblem mit depth-first Search lösen.
Auch hier besteht der Suchraum aus Teillösungen.
Gesucht wird ein Weg vom zu lösenden Sudoku zum 
vollständig ausgefüllten Gitter.

Wie beim Damenproblem definieren wir die Nachbarn einer Teillösung `s`:  
Eine Teillösung `t` ist Nachbar von `s`, falls sich `t` eine Zahl mehr enthält als `s`.

Die am einfachsten zu implementierende Suchstrategie ist,
jeweils eine Teillösung im ersten freien Feld fortzusetzen. 
Eine in vielen Fällen effizientere Strategie ist, 
ein freies Feld zu wählen, in welches möglichst wenig Zahlen eingesetzt werden können.

Wir repräsentieren ein Sudoku als Tuple der Länge 81 mit Zahlen 0-9. Die Null markiert leere Felder.
Die Indizes der Reihen und Spalten und Blöcke ergeben sich so:
- $n$-te Zeile: $9n + i$, ($0\leq i < 9$)
- $n$-te Spalte: $9j + n$, ($0\leq j < 9$)


- $n$-ter Block: Im grossen Gitter der Blöcke hat Block $n$
  die Koordinaten $(X, Y) = (n\%3, n//3)$, und das linke obere Feld diese Blocks hat die Koordinaten
$(x_0, y_0)= (3X, 3Y)$.  
    - erste Zeile: $9y_0  + x_0 + j$, ($0\leq j < 3$)
    - zweite Zeile: $9(y_0+1)  + x_0 + j$, ($0\leq j < 3$)
    - dritte Zeile: $9(y_0+2)  + x_0 + j$, ($0\leq j < 3$)


Hier findet man einige 
[anspruchsvolle Sudokus](https://github.com/nurdymuny/sudoky-energy/blob/master/puzzles/hardest/hardest_puzzles.txt) Sudokus, gegeben als Strings (`hardest_puzzeles.txt` ist eine Kopie des gelinkten Files).

### Aufgabe 1
Erstelle mit Hilfe dieses Notebooks ein Modul `sudoku_solver.py`,
welches eine Funktion `solve_sudoku(sudoku: str) -> str` 
bereitstellt, welche einen Sudoku-String als Argument nimmt und den Sudoku-String der Lösung zurück gibt.
Teste, ob es sich bei Argument tatsächlich um einen Sudoku-String handelt.

Erstellen dann ein Notebook `Test_SudokuSolver.ipynb`.

Schreibe dann eine Funktion, welche das File `hardest_puzzles.txt` liest,
und jeweils das Sudoku mit Beschreibung und Lösung ausgibt.  

### Aufgabe 2
Schreibe eine Funktion, die ein Sudoku mit einer eindeutigen Lösung liefert.

Vorgehen:
1. Fülle die 3 Blocks in der Diagonale mit zufälligen Zahlen.
2. Vervollständige diese Teillösung.
3. Iteriere in zufälliger Reihenfolge über die Indices i in 0..80.
   Solange noch 30 oder mehr Zahlen im Sudoku sind:  
   Setze das Feld i auf 0. Falls die Lösung nicht mehr eindeutig ist,
   setze die entfernte Zahl wieder ein.  

In [ ]:
def search_df(node, get_neighbors):
    nodes_to_visit = [node]
    count = 0

    while nodes_to_visit:
        count += 1
        node = nodes_to_visit.pop()
        if 0 not in node:
            print(f'Nodes visited: {count}')
            return node

        nodes_to_visit.extend(get_neighbors(node))


def search_df_all(node, get_neighbors):
    nodes_to_visit = [node]

    while nodes_to_visit:
        node = nodes_to_visit.pop()
        if 0 not in node:
            yield node

        nodes_to_visit.extend(get_neighbors(node))

In [ ]:
escargot = '100007090030020008009600500005300900010080002600004000300000010040000007007000300'
tarek = '000000000000003085001020000000507000004000100090000000500000073002010000000040009'
hard1 = '400000805030000000000700000020000060000080400000010000000603070500200000104000000'

In [ ]:
def print_sudoku_str(s):
    s = s.replace('0', '.')
    s = '\n'.join(s[9*i:9*i+9] for i in range(9))
    print(s)


def as_tuple(s):
    return tuple(int(c) for c in s)


def tuple2str(sudoku):
    return ''.join(str(n) for n in sudoku)


def print_sudoku(s):
    print_sudoku_str(tuple2str(s))

In [ ]:
print_sudoku_str(escargot)

In [ ]:
sudoku = as_tuple(escargot)
sudoku[:9]

In [ ]:
tuple2str(sudoku) == escargot

In [ ]:
def get_row(sudoku, row):
    return (n for i in range(9)
            if (n := sudoku[9*row+i]) > 0)


def get_col(sudoku, col):
    return (n for j in range(9)
            if (n := sudoku[9*j + col]) > 0)


def get_block(sudoku, blk):
    row = (blk // 3) * 3
    col = (blk % 3) * 3
    return (n for i in range(3) for j in range(3)
            if (n := sudoku[9*(row + i) + col + j]) > 0)

In [ ]:
n = 0
for get_group in (get_row, get_col, get_block):
    name = get_group.__name__[4:]
    numbers = list(get_group(sudoku, n))
    print(f'numbers in {name} {n}: {numbers}')

In [ ]:
def are_unique(items):
    seen = set()
    for item in items:
        if item in seen:
            return False
        seen.add(item)
    return True


def is_sudoku(sudoku):
    for get_group in (get_row, get_col, get_block):
        for i in range(9):
            if not are_unique(get_group(sudoku, i)):
                return False
    return True

In [ ]:
are_unique((1, 2, 3, 1))

In [ ]:
is_sudoku(sudoku)

In [ ]:
def get_options(sudoku, i):
    if 0 != sudoku[i]:
        return ()

    options = []
    row, col = divmod(i, 9)
    block = 3*(row//3) + col//3
    for number in range(1, 10):
        if not (number in get_row(sudoku, row)
                or number in get_col(sudoku, col)
                or number in get_block(sudoku, block)):
            options.append(number)

    return options


def get_neighbors(sudoku):
    '''fuellt erstes freies Feld mit allen moeglichen Zahlen'''
    if 0 not in sudoku:
        return

    i = sudoku.index(0)
    numbers = get_options(sudoku, i)

    sudoku = list(sudoku)
    for number in numbers:
        sudoku[i] = number
        yield tuple(sudoku)


def get_good_neighbors(sudoku):
    '''fuellt Feld mit minimaler Anzahl Optionen mit allen moeglichen Zahlen'''
    best_opts = (0, 10, None)  # index, n_opts, opts

    for i, n in enumerate(sudoku):
        if n != 0:
            continue

        opts = get_options(sudoku, i)
        n_opts = len(opts)
        if n_opts < best_opts[1]:
            best_opts = (i, n_opts, opts)
        if n_opts == 1:
            break

    i, _, numbers = best_opts
    sudoku = list(sudoku)
    for number in numbers:
        sudoku[i] = number
        yield tuple(sudoku)

In [ ]:
print_sudoku(sudoku)

In [ ]:
get_options(sudoku, 55)  # 11

In [ ]:
neighbors = get_good_neighbors(sudoku)  # ebenso mit get_good_neighbors

In [ ]:
sudo = next(neighbors)
print_sudoku(sudo), is_sudoku(sudo)

In [ ]:
solution = search_df(as_tuple(escargot), get_neighbors)
print_sudoku(solution)

In [ ]:
solutions = search_df_all(as_tuple(escargot), get_neighbors)

In [ ]:
print_sudoku(next(solutions))